In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/Grammer.pdf")

documents = loader.load()
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_URL = os.getenv("QDRANT_URL")

In [47]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

chunks[0].metadata


{'producer': 'BCL easyPDF 6.00 (0320)',
 'creator': 'NitroPDF 6.0',
 'creationdate': '2011-12-15T21:54:47+01:00',
 'moddate': '2011-12-15T21:54:47+01:00',
 'source': 'data/Grammer.pdf',
 'total_pages': 6,
 'page': 0,
 'page_label': '1'}

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7510.14it/s]


In [19]:
vector_chunks = embedding_model.embed_documents(
    [chunk.page_content for chunk in chunks]
)

In [20]:
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(url=QDRANT_URL,
api_key=QDRANT_API_KEY)



In [21]:
vector_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    collection_name="enterprise_docs"
)

In [45]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
# question = "Give me Examples of Dependency-based rules"
question = "Tell me about Syntax Dependency-based rules"
# question = "What is Rule Designer Tool?"
retrieved_docs = vector_store.similarity_search(
    question,
    k=3
)
context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

rag_prompt = ChatPromptTemplate.from_template("""
You are an Enterprise Knowledge Assistant.

Answer ONLY from the provided context.

If the answer is not present in the context,
say:

"I don't have enough information about it."

----------------------------

Context:

{context}

----------------------------

Question:

{question}

Answer:
""")

model = init_chat_model(model="groq:llama-3.3-70b-versatile")

formatted = rag_prompt.invoke({
    "context":context,
    "question":question
})

formatted

# res = model.invoke(formatted)

# print(formatted.messages)


chain = rag_prompt | model

res = chain.invoke({
    "context":context,
    "question":question
})

# rag_prompt.messages
print(res.content)


Our syntax allows designing the rules that analyze word-word dependencies in a given phrase. Dependency-based rules are typically more complicated than the rules, based on regular expressions.
